# Árbol de decisión para Estudiantes Marketing con Pipeline + MLflow

Este notebook entrena un modelo de árbol de decisión usando el dataset `estudiantes.csv`.

La diferencia importante frente al notebook anterior es que ahora se guarda en MLflow un `Pipeline` completo:

```text
datos originales → preprocesamiento → árbol de decisión
```

De esta manera, Flask o Streamlit pueden enviar las columnas originales del dataset, sin tener que enviar columnas codificadas manualmente.


## 1. Importación de librerías

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from mlflow.models.signature import infer_signature

import mlflow
import mlflow.sklearn


C:\Users\shomi\OneDrive\Documentos\MAESTRIA\Herramientas de IA\Semana 6\env_semana6\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuración de MLflow



In [2]:
mlflow.set_tracking_uri("http://localhost:9090")
mlflow.set_experiment("estudiantes_arbol_pipeline")


<Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1778903058102, experiment_id='3', last_update_time=1778903058102, lifecycle_stage='active', name='estudiantes_arbol_pipeline', tags={}, trace_location=None, workspace='default'>

## 3. Carga del dataset

In [3]:
df = pd.read_csv("data/estudiantes.csv", delimiter=",")
df.head()


,carrera,modalidad,beca,edad,promedio,asistencias,aprobado
0,Industrial,Presencial,Si,29,5.8,64,Si
1,Industrial,Hibrida,Si,27,6.6,51,No
2,Arquitectura,Presencial,Si,29,8.2,84,Si
3,Economia,Presencial,Si,29,6.6,67,No
4,Economia,Presencial,Si,24,5.1,72,No


In [4]:
print("Filas y columnas:", df.shape)
df.info()


Filas y columnas: (5000, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   carrera      5000 non-null   object 
 1   modalidad    5000 non-null   object 
 2   beca         5000 non-null   object 
 3   edad         5000 non-null   int64  
 4   promedio     5000 non-null   float64
 5   asistencias  5000 non-null   int64  
 6   aprobado     5000 non-null   object 
dtypes: float64(1), int64(2), object(4)
memory usage: 273.6+ KB


## 4. Preparación de datos

La variable objetivo `y` se transforma:

- `yes` → `1`
- `no` → `0`

Además, se elimina `duration` de las variables predictoras porque normalmente se conoce después de realizada la llamada. Por tanto, puede generar fuga de información si se usa para predecir antes de contactar al cliente.


In [5]:
df["aprobado"] = df["aprobado"].map({"Si": 1, "No": 0})

X = df.drop(columns=["aprobado"])
y = df["aprobado"]

print("Columnas originales usadas para entrenamiento:")
print(X.columns.tolist())

print("\nDistribución de la variable objetivo:")
print(y.value_counts())


Columnas originales usadas para entrenamiento:
['carrera', 'modalidad', 'beca', 'edad', 'promedio', 'asistencias']

Distribución de la variable objetivo:
aprobado
0    2950
1    2050
Name: count, dtype: int64


## 5. Identificación de columnas categóricas y numéricas

El pipeline hará automáticamente el `OneHotEncoder` para las columnas categóricas y dejará pasar las columnas numéricas.


In [6]:
columnas_categoricas = X.select_dtypes(include=["object"]).columns.tolist()
columnas_numericas = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Columnas categóricas:")
print(columnas_categoricas)

print("\nColumnas numéricas:")
print(columnas_numericas)


Columnas categóricas:
['carrera', 'modalidad', 'beca']

Columnas numéricas:
['edad', 'promedio', 'asistencias']


In [7]:
print(y)

0       1
1       0
2       1
3       0
4       0
       ..
4995    0
4996    1
4997    0
4998    1
4999    0
Name: aprobado, Length: 5000, dtype: int64


## 6. División de datos

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)


Entrenamiento: (4000, 6)
Prueba: (1000, 6)


## 7. Creación del Pipeline

Este es el punto clave para producción.

El `Pipeline` contiene:

1. `preprocessor`: transforma variables categóricas con `OneHotEncoder`.
2. `modelo`: entrena el árbol de decisión.

Cuando guardamos este `Pipeline` en MLflow, Streamlit podrá enviar datos originales como `job`, `marital`, `education`, etc.


In [9]:
# Parámetros del árbol
'''criterion = "entropy"
max_depth = 3
min_samples_split = 100
min_samples_leaf = 50
random_state = 42

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas),
        ("num", "passthrough", columnas_numericas)
    ]
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("modelo", DecisionTreeClassifier(
            criterion=criterion,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=random_state
        ))
    ]
)

pipeline'''


'criterion = "entropy"\nmax_depth = 3\nmin_samples_split = 100\nmin_samples_leaf = 50\nrandom_state = 42\n\npreprocessor = ColumnTransformer(\n    transformers=[\n        ("cat", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas),\n        ("num", "passthrough", columnas_numericas)\n    ]\n)\n\npipeline = Pipeline(\n    steps=[\n        ("preprocessor", preprocessor),\n        ("modelo", DecisionTreeClassifier(\n            criterion=criterion,\n            max_depth=max_depth,\n            min_samples_split=min_samples_split,\n            min_samples_leaf=min_samples_leaf,\n            random_state=random_state\n        ))\n    ]\n)\n\npipeline'

## 8. Entrenamiento, evaluación y registro en MLflow

El modelo se registrará con el nombre:

```text
Practica 2
```

Luego se podrá cargar desde Streamlit con:

```python
mlflow.sklearn.load_model("models:/estudiantes_arboles/18")
```


In [10]:
# PRIMERA OPCION
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas),
        ("num", "passthrough", columnas_numericas)
    ]
)
# 10  configuraciones
configuraciones = [
    {"criterion": "gini", "max_depth": 3, "min_samples_split": 20, "min_samples_leaf": 5, "random_state": 42},
    {"criterion": "entropy", "max_depth": 5, "min_samples_split": 50, "min_samples_leaf": 10, "random_state": 42},
    {"criterion": "log_loss", "max_depth": 10, "min_samples_split": 100, "min_samples_leaf": 20, "random_state": 42},
    {"criterion": "gini", "max_depth": 20, "min_samples_split": 50, "min_samples_leaf": 10, "random_state": 123},
    {"criterion": "entropy", "max_depth": 5, "min_samples_split": 10, "min_samples_leaf": 5, "random_state": 0},
    {"criterion": "log_loss", "max_depth": 10, "min_samples_split": 50, "min_samples_leaf": 10, "random_state": 42},
    {"criterion": "entropy", "max_depth": 10, "min_samples_split": 50, "min_samples_leaf": 10, "random_state": 42},
    {"criterion": "gini", "max_depth": 10, "min_samples_split": 20, "min_samples_leaf": 5, "random_state": 123},
    {"criterion": "log_loss", "max_depth": 10, "min_samples_split": 10, "min_samples_leaf": 5, "random_state": 42},
    {"criterion": "gini", "max_depth": 5, "min_samples_split": 100, "min_samples_leaf": 50, "random_state": 42},
]

for i, config in enumerate(configuraciones):

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("modelo", DecisionTreeClassifier(
                criterion=config["criterion"],
                max_depth=config["max_depth"],
                min_samples_split=config["min_samples_split"],
                min_samples_leaf=config["min_samples_leaf"],
                random_state=config["random_state"]
            ))
        ]
    )

    with mlflow.start_run(run_name=f"estudiante_arbol__{i}"):

        # entrenar
        pipeline.fit(X_train, y_train)

        # predicciones
        y_pred = pipeline.predict(X_test)

        # métricas
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)

        # log parámetros
        mlflow.log_param("criterion", config["criterion"])
        mlflow.log_param("max_depth", config["max_depth"])
        mlflow.log_param("min_samples_split", config["min_samples_split"])
        mlflow.log_param("min_samples_leaf", config["min_samples_leaf"])
        mlflow.log_param("random_state", config["random_state"])

        # log métricas
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        print(f"Run {i}")
        print("Accuracy:", accuracy)
        
         # guardar modelo
        mlflow.sklearn.log_model(
            sk_model=pipeline,
            artifact_path=f"modelo_{i}",
            registered_model_name="estudiantes_arboles"
        )

2026/05/16 03:20:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/16 03:20:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 0
Accuracy: 0.858


Registered model 'estudiantes_arboles' already exists. Creating a new version of this model...
2026/05/16 03:20:07 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: estudiantes_arboles, version 48
Created version '48' of model 'estudiantes_arboles'.
2026/05/16 03:20:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run estudiante_arbol__0 at: http://localhost:9090/#/experiments/3/runs/6f5505575a3d45c79ec8865d5b393fff
🧪 View experiment at: http://localhost:9090/#/experiments/3
Run 1
Accuracy: 0.851


2026/05/16 03:20:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'estudiantes_arboles' already exists. Creating a new version of this model...
2026/05/16 03:20:11 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: estudiantes_arboles, version 49
Created version '49' of model 'estudiantes_arboles'.
2026/05/16 03:20:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

2026/05/16 03:20:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'estudiantes_arboles' already exists. Creating a new version of this model...
2026/05/16 03:20:39 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: estudiantes_arboles, version 56
Created version '56' of model 'estudiantes_arboles'.
2026/05/16 03:20:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run estudiante_arbol__8 at: http://localhost:9090/#/experiments/3/runs/9eee7cd52eeb4ab8a2dca5bbc39ff282
🧪 View experiment at: http://localhost:9090/#/experiments/3
Run 9
Accuracy: 0.855


2026/05/16 03:20:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'estudiantes_arboles' already exists. Creating a new version of this model...
2026/05/16 03:20:43 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: estudiantes_arboles, version 57


🏃 View run estudiante_arbol__9 at: http://localhost:9090/#/experiments/3/runs/46a20800d7ef4d6aa2eb3401d28c886a
🧪 View experiment at: http://localhost:9090/#/experiments/3


Created version '57' of model 'estudiantes_arboles'.
